The `seqIO` package reads and writes sequence files. `NucSeqIO` streams the
records of a FASTA or FASTQ file as `NucSeqRecord` objects.

In [ ]:
%use biokotlin
import biokotlin.seq.*
import biokotlin.seqIO.*
import biokotlin.seqIO.SeqFormat.*

## Opening a file

Naming the format is optional; it is inferred from the file extension when
omitted. A reader is a single-pass stream, so construct a new one whenever you
need to start over.

In [ ]:
val path = "../src/test/resources/biokotlin/seqIO/B73_Ref_Subset.fa"

// Equivalent to NucSeqIO(path) -- the .fa suffix implies fasta.
NucSeqIO(path, fasta).take(3).forEach { println(it.id) }

Stream every record in the file, numbering them as they arrive:

In [ ]:
NucSeqIO(path).forEachIndexed { index, record ->
    println("$index: ${record.id} (${record.sequence.size()} bp)")
}

## Reading a whole file at once

`readAll` returns every record keyed by its identifier. Use it when the file
is small enough to hold in memory and you need random access.

In [ ]:
val records = NucSeqIO(path).readAll()
println("Read ${records.size} records: ${records.keys}")

Each record wraps a `NucSeq`, so the whole sequence API is available on the
way past.

In [ ]:
val first = records.values.first()
println("id:                 ${first.id}")
println("length:             ${first.sequence.size()} bp")
println("first 60 bp:        ${first.sequence[0..59]}")
println("reverse complement: ${first.sequence[0..59].reverse_complement()}")
println("GC count:           ${first.sequence.gc()}")

## Pulling one record at a time

`read` returns the next record, or `null` once the file is exhausted. This is
the low-level equivalent of iterating, and keeps memory flat over large files.

In [ ]:
val reader = NucSeqIO(path)
var totalBases = 0
while (true) {
    val record = reader.read() ?: break
    totalBases += record.sequence.size()
}
println("Total sequence length: $totalBases bp")